# Video Trails: CSV → deque → frame loop → MP4

Build a video effect from supplied coordinates. Read `Frame`, `Visibility`, `X`, and `Y`, remember recent positions with a deque, and draw on decoded frames. Use frame-key lookup, handle missing positions, export the complete video and add your own design.

**Rendering positions is different from finding the ball.** A CSV can come from manual labels, a simple color rule, or a learned detector. The supplied files state their coordinate source; they are estimates, not ground truth.

Run this notebook from the `take-home-code` folder. Install the optional video dependency in your own environment with `python -m pip install -r requirements.txt` (`python3` on macOS).

In [ ]:
from collections import deque
from pathlib import Path
import csv
import json
import cv2

from render_trail import load_positions, position_at, draw_frame, render_video

VIDEO = Path('../take-home-football-tracker/assets/single-ball.mp4')
CSV = Path('data/soccer.csv')
OUTPUT = Path('outputs/notebook-trail.mp4')
TRAIL_LENGTH = 8


## 1. Inspect the data contract

`Frame` is a **zero-based video frame index**, not the CSV row number. `Visibility=0` means no position is supplied for that frame. `X` points right and `Y` points down in the original video's pixel coordinates. The extra `Source` column explains how a coordinate was obtained.

Read a few rows, count missing positions and compare the CSV range with the video length. A shorter CSV must not crash or truncate the video.

In [ ]:
positions = load_positions(CSV)
print('CSV rows:', len(positions))
print('Visible-position rows:', sum(row.visible for row in positions.values()))
for frame in sorted(positions)[:4]:
    print(positions[frame])

capture = cv2.VideoCapture(str(VIDEO))
assert capture.isOpened(), 'The local video could not be opened.'
fps = capture.get(cv2.CAP_PROP_FPS)
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
capture.release()
print('Video:', frame_count, 'frames,', fps, 'fps,', width, 'x', height)
print('Trail duration:', TRAIL_LENGTH / fps, 'seconds')


## 2. Exercise: keep recent observations

Complete `remember_position`. It should look up `frame_index`, obtain a valid point with `position_at`, and append that point (or `None`) to the **left** of the deque. The deque's `maxlen` removes the oldest item automatically.

Predict the contents after points A, B, a missing frame, and C. A missing frame must stay in history as a gap; dropping it would connect observations across an unseen interval.
<details><summary>After predicting: check the order</summary>

With capacity at least four, the newest item is on the left: `[C, None, B, A]`. `None` occupies a history slot and breaks the connecting line. With capacity three, A has already fallen off: `[C, None, B]`. This changes remembered history, not the coordinates themselves.

</details>


In [ ]:
def remember_position(history, positions, frame_index, width, height):
    # TODO 1: Get the point for this exact frame using position_at().
    # TODO 2: Append it to the left of history, including None for a gap.
    raise NotImplementedError('Complete TODO 1 and TODO 2 before calling this function.')

history = deque(maxlen=TRAIL_LENGTH)
# After completing the function, try it on the first few frame indices.


In [ ]:
# This small check uses synthetic positions, not a camera or video decoder.
# Run the final line after completing remember_position above.
from collections import deque
from render_trail import Position

def check_remember_position(remember):
    examples = {0: Position(0, True, 1, 1),
                1: Position(1, True, 2, 2),
                3: Position(3, True, 3, 3)}
    recent = deque(maxlen=4)
    for frame in range(4):
        remember(recent, examples, frame, 10, 10)
    assert list(recent) == [(3, 3), None, (2, 2), (1, 1)], (
        'Newest belongs on the left. Keep None for missing frame 2.')
    remember(recent, examples, 4, 10, 10)
    assert list(recent) == [None, (3, 3), None, (2, 2)], (
        'One observation per frame; maxlen=4 removes only the oldest item.')
    print('PASS: frame lookup, newest-first order, missing-frame gaps and bounded history.')

# check_remember_position(remember_position)


## 3. Exercise: write your own complete frame loop

Use `capture.read()` until it returns `success=False`. Add one observation per decoded frame, draw it, write the frame, and increment the frame index. Always release both the capture and writer. `VideoWriter` needs the original frame rate and dimensions.

Complete the two functions above/below before calling your version. Compare its result with the working reference in the next section.

In [ ]:
def export_my_version(video_path, csv_path, output_path, trail_length=8):
    positions = load_positions(csv_path)
    capture = cv2.VideoCapture(str(video_path))
    fps = capture.get(cv2.CAP_PROP_FPS)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    history = deque(maxlen=trail_length)
    frame_index = 0
    try:
        while True:
            success, image = capture.read()
            if not success:
                break
            # TODO 3: Call remember_position() for frame_index.
            # TODO 4: Draw the history; do not connect across None entries.
            # TODO 5: Write the image and increment frame_index.
            raise NotImplementedError('Complete TODO 3–5 before rendering your version.')
    finally:
        capture.release()
        writer.release()
    return frame_index


## 4. Run and inspect the working reference

The complete `render_trail.py` solution is provided separately. It uses a dictionary keyed by `Frame`, a bounded deque, explicit line gaps and a `VideoWriter`. It processes every decoded frame even when the CSV ends first.

The exported MP4 uses the available `mp4v` encoder. Check it in a local video player; browser codec support can differ. The local project page also previews the same CSV drawing over the original video without re-encoding it.

In [ ]:
report = render_video(VIDEO, CSV, OUTPUT, trail_length=TRAIL_LENGTH, effect='rock', radius=200)
print(json.dumps(report, indent=2))
assert report['frames_written'] == frame_count
assert abs(report['fps'] - fps) < 0.001


## 5. Change the effect, then change the evidence

Choose at least one visual change and one data/test change:

- Fade older points or make line width depend on speed.
- Use the rock sticker (`--effect rock --radius 200` for soccer), the capture ball, or your own image (`effect="custom", sticker_path="my-sticker.png", sticker_fit="contain"`). Use a transparent PNG and adjust radius to cover the ball.
- Compare different trail lengths in **frames and seconds**.
- Use `measure_orange_ball.py` to generate ping-pong coordinates with a simple color rule. Inspect occlusion and similar-color failures.
- Label several visible and absent frames yourself, or test another video with matching coordinates.

Your portfolio should include your code changes, coordinate provenance, a short before/after result, an output-frame-count check and at least one failure case. A smooth-looking trail is not proof that the coordinates locate the correct object.

The supplied soccer CSV includes actual sampled detections and labeled interpolation from the separate CNN demo. The supplied ping-pong CSV contains per-frame color-component measurements. See `data/provenance.json` and the original media attribution before sharing.

## 6. Optional: compare three tiny CNNs

The browser page can train three small CNNs without downloading a notebook. To run the Python counterpart here, install `requirements-cnn.txt` in addition to the video requirements.

All three predict a center from a 32 × 18 RGB frame. Compare 1,266 / 922 / 1,870 parameters, validation-selected training epochs and held-out coordinate error against a training-mean baseline. Seed 314 and a chronological 60/20/20 visible-frame split keep the comparison consistent.

The supplied coordinates are estimates, not ground truth. These regressors always output a point, including on frames without a visible ball. Test a second independently labeled video before making a general accuracy claim. The recommended longer soccer-juggling clip is `../take-home-football-tracker/assets/soccer-juggling.mp4` with `data/juggling.csv`; it has 346 frames and preserves missing estimates.


In [ ]:
def run_small_cnn_comparison(video_path=VIDEO, csv_path=CSV, epochs=20):
    from tiny_cnn import compare
    return compare(video_path, csv_path, epochs)

# After installing the optional PyTorch dependency, run:
# cnn_report = run_small_cnn_comparison()
# print(json.dumps(cnn_report, indent=2))
